In [3]:
import boto3
from datetime import datetime, timezone
import os
import io
import requests
import json
from app import db_utils, models
import gzip
import msgpack

def load_env_file(filepath=".env"):
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ[key] = value

# Only load environment variables if they're not already set
required_env_vars = ['UPLOAD_ENDPOINT_URL', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'DEV_URL']
if not all(var in os.environ for var in required_env_vars):
    load_env_file()

session = boto3.session.Session()

region_name = 'auto'
endpoint_url = os.environ['UPLOAD_ENDPOINT_URL']
aws_access_key_id = os.environ['AWS_ACCESS_KEY_ID']
aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY']
dev_url = os.environ['DEV_URL']

def write_to_r2(data, relative_path, use_gzip=False):
    client = session.client(
        's3',
        region_name='auto',
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key
    )

    if not use_gzip:
        json_bytes = io.BytesIO(json.dumps(data).encode('utf-8'))
        client.upload_fileobj(json_bytes, 'rshf', relative_path)
        return dev_url + '/' + relative_path


    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode="w") as gz:
        gz.write(json.dumps(data, separators=(",", ":")).encode("utf-8"))
    buf.seek(0) 
    
    client.upload_fileobj(
        buf,
        'rshf',
        relative_path,
        ExtraArgs={           # Helps R2 know what it’s getting
            "ContentType":     "application/json",
            # "ContentEncoding": "gzip"
        }
    )
    return f"{dev_url}/{relative_path}"


def read_from_r2(relative_path):
    resp = requests.get(
        f"{dev_url}/{relative_path}",
        headers={"Accept-Encoding": "identity"},
        timeout=30,
    )
    resp.raise_for_status()

    raw = resp.content
    header_enc = resp.headers.get("Content-Encoding", "").lower()
    looks_gzipped = (
        "gzip" in header_enc
        or (len(raw) >= 2 and raw[0] == 0x1F and raw[1] == 0x8B)
    )

    if looks_gzipped:
        try:
            raw = gzip.decompress(raw)
        except OSError:
            # Not actually gzipped (e.g. already decompressed). Fall through.
            pass

    return json.loads(raw.decode("utf-8"))
    
def write_extension_data_to_r2(db):
    group_memberships = db.query(models.GroupMembership).all()
    accepted_reports = db.query(models.Report).filter(models.Report.accepted == True and models.Report.respondent_role_after == "kicked").all()
    db.close()
    data = dict()

    for obj in group_memberships:
        store_data = [
            obj.cf_handle,
            obj.user_group_rating,
            obj.user_group_max_rating
        ]
        if obj.group_id not in data:
            data[obj.group_id] = dict()
    
        data[obj.group_id][obj.user_id] = store_data


    usrs = [
        "randomp",
        "3bkarm",
        "abhipista_das",
        "Lets_Do_CP",
        "Harshi_2604"
    ]

    for u in usrs:
        data["main"][u] = [
            u,
            -1000000000,
            -1000000000
        ]
    
    for report in accepted_reports:
        data[report.group_id][report.respondent_cf_handle] = [
            report.respondent_cf_handle,
            -1000000000,
            -1000000000
        ]
    
    res = {
        'timestamp': datetime.utcnow().replace(tzinfo=timezone.utc).isoformat(),
        'data': data,
        'data_format': [
            'cf_handle', 'user_group_rating', 'user_group_max_rating'
        ],
    }
    
    extension_data_link = write_to_r2(res, 'extension_data', use_gzip=True)
    timestamp_link = write_to_r2(
        {'timestamp':datetime.utcnow().replace(tzinfo=timezone.utc).isoformat()},
        'timestamp'
    )
    print(f"Finished writing {len(group_memberships)} entries to r2")
    return extension_data_link, timestamp_link

def read_extension_data_from_r2():
    return read_from_r2('extension_data')
    

In [5]:
db = db_utils.SessionLocal()
write_extension_data_to_r2(db)

/var/folders/0p/2_6js4m90bg6mzt9b9kjjm4r0000gn/T/ipykernel_70175/1818498898.py:134: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().replace(tzinfo=timezone.utc).isoformat(),
/var/folders/0p/2_6js4m90bg6mzt9b9kjjm4r0000gn/T/ipykernel_70175/1818498898.py:143: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {'timestamp':datetime.utcnow().replace(tzinfo=timezone.utc).isoformat()},


Finished writing 117861 entries to r2


('https://pub-e98285daadd4482fb56021ad394144c1.r2.dev/extension_data',
 'https://pub-e98285daadd4482fb56021ad394144c1.r2.dev/timestamp')

In [6]:
data = read_extension_data_from_r2()

In [9]:
data["data"]["main"]["3bkarm"]

['3bkarm', -1000000000, -1000000000]

In [ ]:
{
    
}